In [ ]:
# # %pip install matplotlib
# %pip install seaborn




In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns


print("Matplotlib version:", matplotlib.__version__)
print("Seaborn version:", sns.__version__)
print("All libraries loaded successfully!")

In [ ]:
df = pd.read_csv("../data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
df.head(3)

In [ ]:
df.info()

In [ ]:
# Checking missing values 
print(f"Total null values in the emp attrition data: {df.isnull().sum().sum()}")


In [ ]:
# Target value and their percentage and counts
attrition=df["Attrition"]
print(f"Value count of target variable is :\n{attrition.value_counts()}")
print(f"\nPercentage of attrition: \n{attrition.value_counts(normalize=True).mul(100).round(2)}")





EDA-Steps


In [ ]:
#  Eda is performed to understand patterns,
#  in the data, identify outliers, and form hypotheses about relationships between variables.
# for example we want to investigate-
#     are emp who work overtime are more likely to leave?
#     Does montly income differ bw emp who leave and those who stay?
#     How many employees leave compared with those who stay?

sns.countplot(data=df,x="StandardHours",hue="Attrition")
plt.title("Employee Attrition Distribution")
plt.xlabel("Works overtime")
plt.ylabel("Number of Employees")
plt.show()


In [ ]:
df.head()


In [ ]:
# compoare monthly income
sns.boxplot(data=df,x="Attrition",y="MonthlyIncome")
plt.title("Monthly Income vs Employee Attrition")
plt.xlabel("Employee Attrition")
plt.ylabel("Monthly Income")
plt.show


In [ ]:
print(f"{attrition.value_counts()}")

In [ ]:
overtime_attrition=pd.crosstab(df["OverTime"],df["Attrition"],normalize="index")
overtime_attrition_rate=(overtime_attrition*100).round(2)

overtime_attrition_rate
# it shows percentage

In [ ]:
monthlyincome_attrition=pd.crosstab(df["MonthlyIncome"],df["Attrition"],normalize="index")
monthlyincome_attrition_rate=(monthlyincome_attrition*100).round(2)
monthlyincome_attrition_rate

In [ ]:
income_comparison=df.groupby("Attrition")["MonthlyIncome"].median()
income_comparison

In [ ]:
# Difference between mean and median in our actual data
income_summary=df.groupby("Attrition")["MonthlyIncome"].agg(["mean","median"])
print(income_summary.round(2))


In [ ]:
# # Observations till here
# 1-the mean is higher than the median
# 2- this means that some higher salary or income may be pulling the average upwards
# 3-for emp who left also mean is high


In [ ]:
sns.histplot(
    data=df,
    x="MonthlyIncome",
    hue="Attrition",
    multiple="stack",
    kde=True,
    bins=30
)
# hue-"attrition" separates the two groups visually
# bins- divide the income range into 30 intervals..
plt.title("Distribution of Monthly Income by Attrition")
plt.xlabel("Monthly Income")
plt.ylabel("Number of Employees")

plt.show()

<!-- Step 2: EDA findings
Analysis	Our observation	What it tells us
Attrition distribution	1,233 stayed; 237 left	The dataset has class imbalance.
Overtime	Attrition rate is higher among employees who work overtime	Overtime may be associated with attrition.
Monthly income	Median income is higher among employees who stayed	Income may be associated with attrition.


These are observations from the dataset, not proof that overtime or income causes employees to leave. -->

In [ ]:
# Step 2: EDA findings
# Analysis	Our observation	What it tells us
# Attrition distribution	1,233 stayed; 237 left	The dataset has class imbalance.
# Overtime	Attrition rate is higher among employees who work overtime	Overtime may be associated with attrition.
# Monthly income	Median income is higher among employees who stayed	Income may be associated with attrition.


# These are observations from the dataset, not proof that overtime or income causes employees to leave.

In [ ]:
# Step 3: Prepare the data for Machine Learning
print("Number of duplicate rows:", df.duplicated().sum())
y=df["Attrition"]    #This is our target variable
X = df.drop(columns=["Attrition"])
# X,y
X.info()

In [ ]:
y.value_counts()

In [ ]:
# # Observation till now
# So far, we've learned:
# Step	What we checked	          Result
# 1	    Dataset size	           1,470 rows and 35 columns
# 2	    Missing values             0,	You checked them
# 3	    Attrition distribution	    1,233 stayed; 237 left
# 4	    Overtime vs. attrition	    Higher attrition rate among employees working overtime
# 5	    Monthly income	Higher   mean and median among employees who stayed
# 6	    Duplicate rows	0

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2, random_state=42, stratify=y)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
# 3. stratify=y
# This is especially useful for our project.
# Remember that we have:
# - 1,233 employees who stayed.
# - 237 employees who left.
# stratify=y helps preserve approximately the same proportions of Yes and No in both the training and testing sets.
# This is important because our dataset has class imbalance.

In [ ]:
# Step 3.3 — Prepare the data for Machine Learning
# Before training our model, we need to understand one important problem.
# Look at our features:
# Feature	Example	Type
# Age	35	Numerical
# MonthlyIncome	5000	Numerical
# OverTime	Yes	Categorical
# JobRole	Research Scientist	Categorical


# Some features contain numbers, while others contain text.
# Most scikit-learn machine learning models need numerical inputs. Therefore, we must convert categorical features into a numerical representation.
# This process is called encoding.

In [ ]:

numerical_columns=X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
print(f"numerical_columns length= {len(numerical_columns)}")
include_=["object", "category", "bool"]
categorical_columns=X_train.select_dtypes(
    include=include_
).columns.tolist()
print(f"categorical_columns length= {len(categorical_columns)}")


In [ ]:
print("Missing values in X_train:", X_train.isnull().sum().sum())

print("\nMissing values in numerical columns:")
print(X_train[numerical_columns].isnull().sum().sum())

print("\nMissing values in categorical columns:")
print(X_train[categorical_columns].isnull().sum().sum())

Pre-Processing Pipelines-

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Setting Numerical Preprocessing rules
numerical_transformer=Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)
# numerical_transformer
# setting Categorical Preprocessing rules
categorical_transformer=Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)
# Combining both preprocessing tranfromer branches
preprocessor=ColumnTransformer(
    transformers=[
        ("num",numerical_transformer,numerical_columns),
        ("cat",categorical_transformer,categorical_columns)
    ]
)
print("Preprocessing pipeline created successfully!")

Transform the training data

In [ ]:
# X_train_processed=preprocessor.fit_transform(X_train)
# X_train_processed.shape,X_train.shape
# X_test_processed = preprocessor.transform(X_test)
# print("Original test shape:", X_test.shape)
# print("Processed test shape:", X_test_processed.shape)
# if we are using preprocessing pipeline then no neeed of. mannual preprocessing

I am going to build my first Machine Learning Model With full understanding of all the implementation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])



Step 4.3 — Train the model


In [ ]:
model.fit(X_train, y_train)

print("Model training completed successfully!")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Target values:", y_train.value_counts())
# print("Model steps:", model.steps)

Lets make prediction---

In [ ]:
y_pred = model.predict(X_test)

print("First 10 actual values:")
print(y_test.head(10).values)

print("\nFirst 10 predicted values:")
print(y_pred[:10])

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.2%}")

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay
import matplotlib.pyplot as plt
cm=confusion_matrix(
    y_test,y_pred,labels=["No","Yes"]
)
cm
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Stayed (No)", "Left (Yes)"]
)

display.plot()
plt.show()

Second Model is going to create 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

improved_model=Pipeline(
    steps=[
        ("preprocessor",preprocessor),
        ("classifier",LogisticRegression(max_iter=1000,class_weight="balanced"))
    ]
)
print("Balanced Model which is more imporved is created..")

In [ ]:
improved_model.fit(X_train, y_train)

print("Balanced model training completed!")

<!-- 1. Load the dataset
        ↓
2. Understand and explore the data
        ↓
3. Separate features (X) and target (y)
        ↓
4. Split data into training and testing sets
        ↓
5. Identify numerical and categorical columns
        ↓
6. Create the preprocessing pipeline
        ↓
7. Create the complete ML pipeline
        ↓
8. Train the model using X_train and y_train
        ↓
9. Predict using X_test
        ↓
10. Evaluate the model -->

In [ ]:
y_pred_improved = improved_model.predict(X_test)

print(f"actual\n{y_test.head(10).values}")
print()
print(f"improved prediction:\n{y_pred_improved[:10]}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Accuracy:", f"{accuracy_score(y_test, y_pred_improved):.2%}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_improved))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_improved, labels=["No", "Yes"]))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

models = {
    "Original Logistic Regression": y_pred,
    "Improved Logistic Regression": y_pred_improved
}

results = []

for model_name, predictions in models.items():
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision (Yes)": precision_score(
            y_test, predictions, pos_label="Yes"
        ),
        "Recall (Yes)": recall_score(
            y_test, predictions, pos_label="Yes"
        ),
        "F1-score (Yes)": f1_score(
            y_test, predictions, pos_label="Yes"
        )
    })

comparison_df = pd.DataFrame(results)

comparison_df.round(3)

In [ ]:
yes_idx = list(model.classes_).index("Yes")   # model = your original LogisticRegression
probs = model.predict_proba(X_test)[:, yes_idx]

In [ ]:
import numpy as np

rows = []
for t in [0.5, 0.4, 0.35, 0.3, 0.25, 0.2]:
    preds = np.where(probs >= t, "Yes", "No")
    rows.append({
        "Threshold": t,
        "Precision": precision_score(y_test, preds, pos_label="Yes"),
        "Recall": recall_score(y_test, preds, pos_label="Yes"),
        "F1": f1_score(y_test, preds, pos_label="Yes"),
    })

pd.DataFrame(rows).round(3)

In [ ]:
# Get predicted probabilities for the test employees
y_probability = model.predict_proba(X_test)

print("First 5 probability predictions:")
print(y_probability[:5])

print("\nClass order:")
print(model.classes_)

In [ ]:
# Get the probability of leaving (Yes)
y_probability_yes = model.predict_proba(X_test)[:, 1]

# Predict Yes when the probability is at least 0.3
threshold = 0.3

y_pred_03 = ["Yes" if probability >= threshold else "No"
             for probability in y_probability_yes]

# Count the predictions
import pandas as pd

print(pd.Series(y_pred_03).value_counts())

In [ ]:
# Remember this rule:
# - [:, 0] → All employees' probabilities of staying.
# - [:, 1] → All employees' probabilities of leaving.
print("Both columns:")
print(model.predict_proba(X_test)[:5])

print("\nOnly No probabilities:")
print(model.predict_proba(X_test)[:5, 0])

print("\nOnly Yes probabilities:")
print(model.predict_proba(X_test)[:5, 1])

In [ ]:
# Get probability of leaving for every employee
y_probability_yes = model.predict_proba(X_test)[:, 1]
# Predictions using the default threshold: 0.5
y_pred_05 = ["Yes" if p >= 0.5 else "No"
             for p in y_probability_yes]
# Predictions using a lower threshold: 0.3
y_pred_03 = ["Yes" if p >= 0.3 else "No"
             for p in y_probability_yes]




In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Threshold": [0.5, 0.3],
    "Predicted No": [
        y_pred_05.count("No"),
        y_pred_03.count("No")
    ],
    "Predicted Yes": [
        y_pred_05.count("Yes"),
        y_pred_03.count("Yes")
    ]
})

print(comparison)

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report
# Threshold 0.5
# y_pred_05
# y_pred_03
cm=confusion_matrix(y_test,y_pred_05,labels=["No","Yes"])
print(cm)
print(classification_report(y_test,y_pred_05,labels=["No","Yes"]))

In [ ]:
# cm_03
# y_pred_03
cm_03=confusion_matrix(y_test,y_pred_03,labels=["No","Yes"])
print(classification_report(y_test,y_pred_03,labels=["No","Yes"]))

In [ ]:
y_test_prob_no=model.predict_proba(X_train)
y_test_prob_no=y_test_prob_no[:,1]  #this is the probablility of yes (i.e leaving of every employee)
y_test_prob_no[:5]

In [ ]:
from sklearn.metrics import accuracy_score

# Step 1: Predict on training data
y_train_pred = model.predict(X_train)

# Step 2: Predict on test data
y_test_pred = model.predict(X_test)

# Step 3: Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Step 4: Compare results
print("Training Accuracy:", round(train_accuracy, 4))
print("Testing Accuracy:", round(test_accuracy, 4))
print("Difference:", round(train_accuracy - test_accuracy, 4))

In [ ]:
from sklearn.model_selection import cross_val_score,StratifiedKFold
# Create 5 folds
cv=StratifiedKFold(n_splits=5,
                   shuffle=True,
                   random_state=42
                   )
cv
cv.get_metadata_routing
# evaluate the model using cross-validation
cv_scores=cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

# Display the results
print("Scores for each fold:", cv_scores)
print(f"Average CV accuracy: {cv_scores.mean():.2%}")
print(f"Standard deviation: {cv_scores.std():.2%}")

In [ ]:
print(model.named_steps["classifier"])

In [ ]:
print("Current C:", model.named_steps["classifier"].C)

Model HyperParameter Tuning-

In [ ]:
# We have five values to test: { 0.01, 0.1, 1, 10, and 100.}
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}
f1_yes_scorer = make_scorer(
    f1_score,
    pos_label="Yes"
)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring=f1_yes_scorer,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

In [ ]:
print("Best C:", grid_search.best_params_["classifier__C"])

print(
    "Best Cross-Validation F1-Score:",
    round(grid_search.best_score_, 3)
)

In [ ]:
# Lets save the best model selected by GridSearch
best_model = grid_search.best_estimator_

print("Best model:")
print(best_model.classes_)


In [ ]:
from sklearn.model_selection import cross_val_score

best_cv_scores = cross_val_score(
    best_model,
    X_train,
    y_train,
    cv=cv,
    scoring=f1_yes_scorer,
    n_jobs=-1
)

print("F1-score for each fold:", best_cv_scores)
print(f"Average CV F1-score: {best_cv_scores.mean():.3f}")
print(f"Standard deviation: {best_cv_scores.std():.3f}")

In [ ]:
print("Original model C:", model.named_steps["classifier"].C)

print(
    "Tuned model C:",
    best_model.named_steps["classifier"].C
)

print(
    "Original CV F1:",
    round(
        cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring=f1_yes_scorer,
            n_jobs=-1
        ).mean(),
        3
    )
)

print("Tuned CV F1:", round(best_cv_scores.mean(), 3))

Step 2: Evaluate your best model on test data

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

# Make predictions on the test dataset
y_pred_final = best_model.predict(X_test)

# Evaluate the model
print("Final Test Results")
print("------------------")

print(f"Accuracy:  {accuracy_score(y_test, y_pred_final):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_final, pos_label='Yes'):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_final, pos_label='Yes'):.3f}")
print(f"F1-score:  {f1_score(y_test, y_pred_final, pos_label='Yes'):.3f}")
cm = confusion_matrix(
    y_test,
    y_pred_final,
    labels=["No", "Yes"]
)

print("Confusion Matrix:")
print(cm)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred_final,
    labels=["No", "Yes"]
)

print("Confusion Matrix:")
print(cm)

In [ ]:
# 2. What are we checking?
# Imagine you're preparing for an exam.
# Situation	            Training performance	Test performance	Meaning
# Overfitting	        Very high	            Much lower	        Model may have learned training-specific patterns
# Underfitting	        Low	                    Low	                Model may not have learned enough
# Good generalization	Good	                Similar	            Model performs similarly on unseen dat

In [ ]:
# finding accuracty of my best model
print(accuracy_score(y_test, y_pred_final))
pirnt


In [ ]:
# 1. Load the dataset
#         ↓
# 2. Understand and explore the data
#         ↓
# 3. Separate features (X) and target (y)
#         ↓
# 4. Split data into training and testing sets
#         ↓
# 5. Identify numerical and categorical columns
#         ↓
# 6. Create the preprocessing pipeline
#         ↓
# 7. Create the complete ML pipeline
#         ↓
# 8. Train the model using X_train and y_train
#         ↓
# 9. Predict using X_test
#         ↓
# 10. Evaluate the model

In [ ]:
# Original Dataset: 1,470 employees
#                  |
#                  v
#          Separate X and y
#                  |
#                  v
#           Split the data
#                  |
#          +-------+-------+
#          |               |
#          v               v
#       X_train           X_test
#       y_train           y_test
#          |               |
#          v               |
#     model.fit()          |
#     Learn patterns       |
#          |               |
#          v               v
#   Predict on X_train   Predict on X_test
#          |               |
#          v               v
#    y_train_pred        y_test_pred
#          |               |
#          v               v
#  Compare with y_train  Compare with y_test
#          |               |
#          v               v
#  Training accuracy      Test accuracy